# Proyecto 3 CUDA — corridas en Colab T4

Segunda tabla, aparte de la RTX 3050. Entorno de ejecución: **GPU T4**.

In [ ]:
!nvidia-smi

## 1. Código y dependencias
No se reinstala TensorFlow: se usa el de Colab (anotar su versión).

In [ ]:
!git clone https://github.com/DiegoOF07/p3-paralela.git /content/p3-paralela
%cd /content/p3-paralela
!pip install -q pyarrow psutil

## 2. Datos en el disco local de Colab
Suban a Drive `asl_alphabet_train.zip` y `splits.pkl` (y `label_map.pkl` si lo tienen). Se copian a `/content`; no se leen directo desde Drive. Ajusten `DRIVE_DIR`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/p3-paralela'  # <- ajustar
!mkdir -p /content/data/asl
!cp "$DRIVE_DIR/splits.pkl" /content/data/asl/
!cp "$DRIVE_DIR/label_map.pkl" /content/data/asl/ 2>/dev/null || true
!unzip -q "$DRIVE_DIR/asl_alphabet_train.zip" -d /content/data/asl/
!ls /content/data/asl/asl_alphabet_train/asl_alphabet_train | head

## 3. Variables de entorno (antes de importar `src`)

In [ ]:
import os
os.environ['ASL_DATA_DIR'] = '/content/data/asl'
os.environ['MACHINE_NAME'] = 'colab-t4'

import tensorflow as tf
print('TensorFlow', tf.__version__)
print(tf.config.list_physical_devices('GPU'))

## 4. Fase 1: preparación de los .npy (secuencial vs paralelo)

In [ ]:
!python -m src.data_prep

## 5. Diagnóstico: solo datos vs solo matemática

In [ ]:
!python -m src.diagnose

## 6. Etapas GPU
Si falta tiempo, agreguen `--reps 1`. `bench.py` se puede volver a ejecutar para retomar.

In [ ]:
!python -m src.bench --stages E1 E3 E4 E5 E6

## 7. Guardar resultados en Drive

In [ ]:
!mkdir -p "$DRIVE_DIR/resultados-colab"
!cp -r results evidence "$DRIVE_DIR/resultados-colab/"
import pandas as pd
runs = pd.read_csv('results/runs.csv')
runs[runs.maquina == 'colab-t4'].tail(20)